# **Vibe experiment: audio features + genres + LLM**

The main analysis (`moodtape_analysis.ipynb`) showed that the nearest neighbours on the 9 audio
features sound alike on paper but often feel like a different vibe: "Brooklyn Baby" got dubstep,
emo and show tunes. Audio features describe *how* a track sounds, not its style or context.

This notebook tests whether a language model's general knowledge about music closes that gap.
For each test case, three ways of recommending are compared:

| Variant | Target point | Candidates | Final ranking |
|---|---|---|---|
| **A. Audio only** | audio features | all tracks | distance |
| **B. Audio + genres** | audio features | tracks in genres the LLM picked from the dataset's 114 genres | distance |
| **C. Audio + genres + LLM pick** | audio features | 50 closest from B | the LLM picks 10 and explains why |

The target point is the reference track's real audio features when it is in the catalog. When it is
not (or when the input is a free-text vibe), the LLM estimates the 9 feature values.

The LLM only **chooses among real tracks from the catalog**: it never suggests tracks of its own,
so every result exists in the dataset.

**Model:** Google Gemini (`gemini-3.5-flash`, free tier). Responses are cached in
`data/gemini_cache.json` (git-ignored), so re-running the notebook does not call the API again.

**Evaluation:** there is no ground truth for "same vibe". The results are judged by a listener who
knows the reference tracks.

In [ ]:
import json
import time
import urllib.error
import urllib.request
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.neighbors import NearestNeighbors

pd.set_option('display.max_colwidth', 80)
DATA_DIR = Path('data')

tracks = pd.read_parquet(DATA_DIR / 'tracks_clean.parquet')
scaler = json.loads((DATA_DIR / 'scaler.json').read_text())
FEATURES = scaler['features']
vectors = np.stack(tracks['features_scaled'].to_numpy())
nn = NearestNeighbors(metric='euclidean').fit(vectors)

ALL_GENRES = sorted({g for genres in tracks['genres'] for g in genres})


def read_env(path='../.env'):
    """Minimal .env reader (KEY=value lines), to avoid an extra dependency."""
    env = {}
    for line in Path(path).read_text().splitlines():
        if '=' in line and not line.lstrip().startswith('#'):
            key, value = line.split('=', 1)
            env[key.strip()] = value.strip()
    return env


API_KEY = read_env().get('GEMINI_API_KEY')
assert API_KEY, "Set GEMINI_API_KEY in the .env file at the project root"
print(f"{len(tracks):,} tracks, {len(ALL_GENRES)} genres")

## **Calling the model**

Gemini's structured output is used: the request includes a JSON schema and the response is
guaranteed to follow it, so no free text has to be parsed. The allowed genres are listed in the
prompt, and any genre not in the dataset is dropped in code (the API rejects an `enum` with all 114
genres). Temperature 0 makes answers as repeatable as possible.

In [ ]:
MODEL = 'gemini-3.5-flash'
CACHE_PATH = DATA_DIR / 'gemini_cache.json'
cache = json.loads(CACHE_PATH.read_text(encoding='utf-8')) if CACHE_PATH.exists() else {}


def ask_gemini(prompt, schema, retries=5):
    """Cached Gemini call with a JSON response schema. Returns the parsed JSON."""
    cache_key = json.dumps([MODEL, prompt, schema], sort_keys=True)
    if cache_key in cache:
        return cache[cache_key]
    body = json.dumps({
        'contents': [{'parts': [{'text': prompt}]}],
        'generationConfig': {'temperature': 0, 'responseMimeType': 'application/json',
                             'responseSchema': schema},
    }).encode()
    url = f'https://generativelanguage.googleapis.com/v1beta/models/{MODEL}:generateContent'
    request = urllib.request.Request(url, data=body, method='POST', headers={
        'Content-Type': 'application/json', 'x-goog-api-key': API_KEY})
    for attempt in range(retries):
        try:
            with urllib.request.urlopen(request, timeout=120) as response:
                data = json.load(response)
            break
        except urllib.error.HTTPError as error:
            if error.code in (429, 500, 503) and attempt < retries - 1:
                time.sleep(10 * (attempt + 1))   # free tier: rate limits and busy servers
                continue
            raise RuntimeError(json.load(error)['error']['message']) from None
    result = json.loads(data['candidates'][0]['content']['parts'][0]['text'])
    cache[cache_key] = result
    CACHE_PATH.write_text(json.dumps(cache, ensure_ascii=False), encoding='utf-8')
    return result

## **Step 1: describing the reference**

The model receives a reference (a track or a free-text vibe) and returns:

* a short description of its style and mood;
* 3 to 8 compatible genres from the dataset's list;
* an estimate of the 9 audio features, on the same scales as the dataset.

For tracks in the catalog, the estimate is compared with the real values: it shows how reliable the
estimates are for references that are *not* in the catalog.

In [ ]:
FEATURE_GUIDE = '''\
danceability 0-1, energy 0-1, loudness in dB (about -60 to 0, typical songs -12 to -4),
speechiness 0-1 (above 0.66 is mostly spoken word), acousticness 0-1, instrumentalness 0-1
(above 0.5 is mostly instrumental), liveness 0-1 (above 0.8 is a live recording),
valence 0-1 (musical positiveness: low is sad or dark, high is happy), tempo in BPM'''

PROFILE_SCHEMA = {
    'type': 'OBJECT',
    'properties': {
        'description': {'type': 'STRING'},
        'genres': {'type': 'ARRAY', 'items': {'type': 'STRING'}},
        'features': {'type': 'OBJECT',
                     'properties': {f: {'type': 'NUMBER'} for f in FEATURES},
                     'required': FEATURES},
    },
    'required': ['description', 'genres', 'features'],
}


def describe(reference):
    prompt = f'''You help recommend music by vibe. Reference: {reference}

1. Describe its style and mood in one or two sentences (do not quote lyrics).
2. Pick 3 to 8 genres, only from this list, whose tracks a fan of this reference would enjoy as
   the same vibe. Prefer specific genres over broad ones.
   Allowed genres: {', '.join(ALL_GENRES)}
3. Estimate its Spotify-style audio features: {FEATURE_GUIDE}'''
    profile = ask_gemini(prompt, PROFILE_SCHEMA)
    invalid = [g for g in profile['genres'] if g not in ALL_GENRES]
    if invalid:
        print(f"Dropped genres not in the dataset: {invalid}")
    return {**profile, 'genres': [g for g in profile['genres'] if g in ALL_GENRES]}


def find_track(name, artist):
    match = tracks[(tracks['track_name'] == name) & tracks['artists'].apply(lambda a: artist in a)]
    return match.index[0] if len(match) else None


def standardize(features):
    raw = np.array([features[f] for f in FEATURES])
    return (raw - scaler['mean']) / scaler['scale']

## **Step 2: the three variants**

In [ ]:
POOL_SIZE = 50

PICK_SCHEMA = {
    'type': 'ARRAY',
    'items': {'type': 'OBJECT',
              'properties': {'id': {'type': 'INTEGER'}, 'reason': {'type': 'STRING'}},
              'required': ['id', 'reason']},
}


def closest(target, candidates, exclude, n):
    """Indices of the n candidate tracks closest to target, skipping excluded ones."""
    candidates = np.setdiff1d(candidates, exclude)
    distances = np.linalg.norm(vectors[candidates] - target, axis=1)
    order = np.argsort(distances)[:n]
    return candidates[order], distances[order]


def as_table(indices, distances=None, reasons=None):
    table = tracks.loc[indices, ['track_name', 'artists', 'genres']].copy()
    table['artists'] = table['artists'].str.join(', ')
    table['genres'] = table['genres'].str.join(', ')
    if distances is not None:
        table['distance'] = np.round(distances, 2)
    if reasons is not None:
        table['why'] = reasons
    return table.reset_index(drop=True)


def llm_pick(reference, profile, pool, n=10):
    lines = '\n'.join(
        f"{i}. {tracks.at[i, 'track_name']} - {', '.join(tracks.at[i, 'artists'])} "
        f"[{', '.join(tracks.at[i, 'genres'])}]" for i in pool)
    prompt = f'''You help recommend music by vibe. Reference: {reference}
Its vibe: {profile['description']}

Candidates (id. track - artists [genres]), all close to the reference on audio features:
{lines}

Pick the {n} candidates that best match the reference's vibe, best first. Use what you know about
the tracks and artists; if you do not know a track, judge by artist and genre. For each, give its id
and a reason of at most 15 words. Do not quote lyrics. Use only ids from the list.'''
    picks = ask_gemini(prompt, PICK_SCHEMA)
    valid = [p for p in picks if p['id'] in set(pool)][:n]
    return [p['id'] for p in valid], [p['reason'] for p in valid]


def run_case(reference, track=None, n=10):
    """reference: text given to the model; track: (name, artist) if it may be in the catalog."""
    profile = describe(reference)
    idx = find_track(*track) if track else None
    in_catalog = idx is not None
    target = vectors[idx] if in_catalog else standardize(profile['features'])
    exclude = [] if not in_catalog else tracks.index[
        (tracks['track_name'] == track[0]) & tracks['artists'].apply(lambda a: track[1] in a)]

    print(f"Reference: {reference}")
    print(f"In catalog: {'yes' if in_catalog else 'no, using the estimated features'}")
    print(f"LLM description: {profile['description']}")
    print(f"LLM genres: {', '.join(profile['genres'])}")
    if in_catalog:
        comparison = pd.DataFrame({'real': tracks.loc[idx, FEATURES].astype(float),
                                   'estimated': pd.Series(profile['features'])[FEATURES]})
        display(comparison.T.round(2))

    a_idx, a_dist = closest(target, tracks.index.to_numpy(), exclude, n)
    in_genres = tracks.index[tracks['genres'].apply(lambda g: bool(set(g) & set(profile['genres'])))]
    pool, pool_dist = closest(target, in_genres.to_numpy(), exclude, POOL_SIZE)
    c_idx, reasons = llm_pick(reference, profile, list(pool), n)

    print("\nA. Audio only")
    display(as_table(a_idx, a_dist))
    print("B. Audio + genres")
    display(as_table(pool[:n], pool_dist[:n]))
    print("C. Audio + genres + LLM pick")
    display(as_table(c_idx, reasons=reasons))

## **Case 1: "Brooklyn Baby" (Lana Del Rey)**

In [ ]:
run_case('the song "Brooklyn Baby" by Lana Del Rey', ('Brooklyn Baby', 'Lana Del Rey'))

## **Case 2: "The Sweet Escape" (Gwen Stefani)**

Not in the catalog (the dataset only has Gwen Stefani duets and No Doubt tracks), so the target point
is the model's estimate of its audio features.

In [ ]:
run_case('the song "The Sweet Escape" by Gwen Stefani feat. Akon', ('The Sweet Escape', 'Gwen Stefani'))

## **Case 3: "Poltergeist" (BANKS)**

In [ ]:
run_case('the song "Poltergeist" by BANKS', ('Poltergeist', 'BANKS'))

## **Case 4: a free-text vibe**

No reference track: the target point comes entirely from the model's estimate.

In [ ]:
run_case('a vibe described by a listener: "melancholic but danceable, for a late-night drive"')

## **Findings**

Results were judged by a listener who knows the three reference tracks.

**Audio only (A) fails in every case.** The closest tracks on audio features cross styles freely:
dubstep and show tunes for "Brooklyn Baby", schlager, salsa and forró for "The Sweet Escape",
country and Mandopop for "Poltergeist", hardcore and blues for the late-night-drive vibe.

**Genres alone (B) are not enough.** The dataset's genre labels are noisy: a Stevie Wonder
Christmas single is labelled `indie`, and the `party` genre filled "The Sweet Escape" with German
party schlager. Filtering by genre removes the worst matches but does not make the list coherent.

**Audio + genres + LLM pick (C) produced coherent lists in all four cases**, for example:

| Reference | Picks from variant C |
|---|---|
| "Brooklyn Baby" | Princess Chelsea, Matt Maltese, Gracie Abrams, Stephen Sanchez |
| "The Sweet Escape" | Sean Kingston & Justin Bieber, Jay Sean, Lady Gaga, Dua Lipa |
| "Poltergeist" | FKA twigs, Blue Foundation, Zero 7, Lianne La Havas |
| "late-night drive" | Chromatics, Electric Youth, RÜFÜS DU SOL, Videoclub |

The listener's verdict: the picks **match, but differently from a streaming service**. Most tracks
were unfamiliar; the familiar ones made sense but would not have been their first thought.
Streaming services recommend from listening behaviour ("people who listen to X also listen to Y"),
which favours the obvious neighbours. Here candidates come from how a track sounds and are filtered
by style, so the results lean towards **discovery**.

**Estimated features are rough.** For tracks in the catalog, the model's estimates missed some
values by a wide margin: valence 0.35 vs a real 0.09 for "Brooklyn Baby"; acousticness 0.15 vs 0.72
and energy 0.65 vs 0.38 for "Poltergeist". The final picks were still coherent, because the genre
filter and the model's choice correct for a target point that is only approximately right. For
references in the catalog, the real features are always used.

**Limitations of this experiment:**

* one listener and four cases: an indication, not a measurement;
* the model knows popular tracks better, so variant C may favour them over obscure ones;
* answers can change between model versions even at temperature 0 (the cache keeps this run
  reproducible);
* the free tier has rate limits, and the model's availability can change.